# Notebook 02_GPU: All-in-One GPU Model Training Pipeline

**Objective**: Consolidated Google Colab GPU training notebook for researchers who prefer executing all deep models (CNN, Fixed GNN, Adaptive GNN) sequentially in a single session.

### Models Trained:
1. **1D CNN Baseline** (`OpportunityCNN`)
2. **Fixed Anatomical GNN** (`FixedAnatomicalGNN`)
3. **Activity-Adaptive Anatomical GNN** (`ActivityAdaptiveAnatomicalGNN`)

> **Compute Requirement**: CUDA GPU (e.g. Google Colab T4 GPU). Select *Runtime > Change runtime type > T4 GPU*.

## 0. Environment Setup & Repository Bootstrap

In [ ]:
# ==============================================================================
# 1. ENVIRONMENT DETECTION & REPOSITORY BOOTSTRAP
# ==============================================================================
import sys
import os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_DIR = Path("/content/Opportunity_HAR")
    if not REPO_DIR.exists():
        print("[INFO] Cloning Opportunity_HAR repository from GitHub...")
        !git clone https://github.com/DhruvikaRajput/Opportunity_HAR.git {REPO_DIR}
    else:
        print(f"[INFO] Repository already present at {REPO_DIR}. Pulling latest changes...")
        !git -C {REPO_DIR} pull origin main

    # Set repository root at front of sys.path
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))

    os.chdir(REPO_DIR)
    !pip install -q -r requirements.txt
else:
    current = Path.cwd()
    REPO_DIR = current.parent if current.name == "notebooks" else current
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    os.chdir(REPO_DIR)

print(f"Active Working Directory : {Path.cwd()}")

# Ensure fresh import of src.api
import importlib
if "src.api" in sys.modules:
    import src.api
    importlib.reload(src.api)
else:
    import src.api as api

import src.api as api
status = api.setup_project()
print(f"High-Level API Loaded    : src.api (locate_dataset available: {hasattr(api, 'locate_dataset')})")
print(f"Compute Device Detected  : {status['device_name']} (CUDA Available: {status['cuda_available']})")

## 1. Configure Experiment Mode

In [ ]:
# Choose: 'FAST' (5 epochs GPU) or 'FULL' (50 epochs early stopping GPU)
EXPERIMENT_MODE = "FAST" if env_info["cuda_available"] else "DEBUG"
print(f"Active Training Mode: {EXPERIMENT_MODE}")
if not env_info["cuda_available"] and EXPERIMENT_MODE != "DEBUG":
    print("Notice: CUDA is not active. Full neural training requires a GPU. In Colab: Runtime > Change runtime type > T4 GPU.")

## 2. Load Processed Data & Prepare DataLoaders

In [ ]:
try:
    subset = api.load_preprocessed_subset("opportunity_locomotion_subset.npz")
    X_train, y_train = subset["X_train"], subset["y_train"]
    X_val, y_val = subset["X_val"], subset["y_val"]
    X_test, y_test = subset["X_test"], subset["y_test"]

    if EXPERIMENT_MODE == "DEBUG":
        X_train, y_train = X_train[:64], y_train[:64]
        X_val, y_val = X_val[:32], y_val[:32]
        X_test, y_test = X_test[:32], y_test[:32]

    train_loader, val_loader, test_loader = api.create_dataloaders(
        X_train=X_train, y_train=y_train,
        X_val=X_val, y_val=y_val,
        X_test=X_test, y_test=y_test,
        batch_size=64 if EXPERIMENT_MODE != "DEBUG" else 16,
    )
    print(f"DataLoaders Ready: {len(train_loader)} train batches, {len(val_loader)} val batches.")
except FileNotFoundError:
    print("[NOTICE] Preprocessed dataset not found. Run Notebook 02 to generate opportunity_locomotion_subset.npz.")

## 3. Train 1D CNN Baseline on GPU

In [ ]:
cnn_model = api.build_cnn(in_channels=133, num_classes=5, sequence_length=30)
try:
    best_cnn, cnn_hist = api.train_model(
        cnn_model, train_loader, val_loader,
        mode=EXPERIMENT_MODE, lr=1e-3, experiment_name="all_in_one_cnn"
    )
    cnn_metrics = api.evaluate_model(best_cnn, test_loader)
    print(f"1D CNN Test Macro F1: {cnn_metrics['macro_f1']*100:.2f}%")
except Exception as e:
    print("CNN Training Notice:", e)

## 4. Train Fixed Anatomical GNN on GPU

In [ ]:
fixed_gnn = api.build_fixed_gnn(num_classes=5, sequence_length=30)
try:
    best_fgnn, fgnn_hist = api.train_model(
        fixed_gnn, train_loader, val_loader,
        mode=EXPERIMENT_MODE, lr=1e-3, experiment_name="all_in_one_fixed_gnn"
    )
    fgnn_metrics = api.evaluate_model(best_fgnn, test_loader)
    print(f"Fixed GNN Test Macro F1: {fgnn_metrics['macro_f1']*100:.2f}%")
except Exception as e:
    print("Fixed GNN Training Notice:", e)

## 5. Train Activity-Adaptive Anatomical GNN on GPU

In [ ]:
adaptive_gnn = api.build_adaptive_gnn(num_classes=5, sequence_length=30)
try:
    best_agnn, agnn_hist = api.train_model(
        adaptive_gnn, train_loader, val_loader,
        mode=EXPERIMENT_MODE, lr=1e-3, experiment_name="all_in_one_adaptive_gnn"
    )
    agnn_metrics = api.evaluate_model(best_agnn, test_loader)
    print(f"Adaptive GNN Test Macro F1: {agnn_metrics['macro_f1']*100:.2f}%")
except Exception as e:
    print("Adaptive GNN Training Notice:", e)

## 6. Inspect Saved Artifacts

In [ ]:
print("Training Complete!")
print(f"Checkpoints saved in: {REPO_DIR / 'checkpoints'}")
print(f"Results saved in    : {REPO_DIR / 'results'}")